# AJOU-VIS 음성 서버 (Colab GPU)

STT · RAG 서버(Claude) · TTS를 **Colab GPU**에서 돌리고, **Mac에서는 마이크·스피커만** 연결한다.

```
[Mac] speech-to-speech talk  ◀── cloudflared 터널 ──▶  [Colab] speech-to-speech serve + RAG 서버
```

**처음 한 번만**
1. 런타임 → 런타임 유형 변경 → **GPU (A100 권장)**
2. 왼쪽 🔑 **보안 비밀(Secrets)** 에 `ANTHROPIC_API_KEY` 추가하고 "노트북 액세스" 켜기

**실행**: 1번 셀(설정)을 확인한 뒤 위에서부터 차례로 실행 → 마지막에 나오는 명령을 Mac 터미널에 붙여넣기.

Mac에서는 `~/speech-to-speech/run_talk_colab.sh <주소>` 로 접속한다 (헤드폰 권장).

In [ ]:
# 1. 설정 — 실험할 조합을 여기서 고른다
STT_BACKEND = "whisper"            # "whisper" 또는 "qwen3-asr"
WHISPER_MODEL = "openai/whisper-large-v3-turbo"
QWEN3_ASR_MODEL = "Qwen/Qwen3-ASR-0.6B-hf"   # 더 정확: "Qwen/Qwen3-ASR-1.7B-hf"

LLM_MODEL = "claude-haiku-4-5"     # 개발·테스트용 (교수님 피드백). 시연 때는 "claude-opus-5"
LLM_EFFORT = "low"

TTS_QUANT = "Q8_0"                 # 3주차 벤치마크 확정값. Q4_K_M은 크래시로 제외
TTS_SPEAKER = "Sohee"

PIPELINE_BRANCH = "main"           # speech-to-speech 포크 브랜치
RAG_BRANCH = "main"                # ajou-vis 브랜치

In [ ]:
# 2. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. 레포 클론 + 파이프라인 설치 (5~10분)
%cd /content
!rm -rf speech-to-speech ajou-vis
!git clone -q -b {PIPELINE_BRANCH} https://github.com/AJOU-VIS/speech-to-speech.git
!git clone -q -b {RAG_BRANCH} https://github.com/AJOU-VIS/ajou-vis.git
!apt-get -qq install -y libportaudio2 libsndfile1 > /dev/null
!pip install -q -e /content/speech-to-speech

# Qwen3-TTS GGML 바이너리가 이 환경(CUDA·glibc)에서 로드되는지 확인
import subprocess, sys
CHECK = (
    "import importlib, importlib.metadata as md\n"
    "files = md.files('qwentts-cpp-python') or []\n"
    "mods = {str(f).split('/')[0].split('.')[0] for f in files if str(f).endswith(('.py', '.so')) and '.dist-info' not in str(f)}\n"
    "for m in mods: importlib.import_module(m)\n"
    "print('GGML 바이너리 로드 OK:', sorted(mods))\n"
)
r = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-500:])
ok = r.returncode == 0
if not ok:
    print("기본 GGML 바이너리 로드 실패 → CUDA 12.4용으로 재설치")
    !pip install -q "qwentts-cpp-python==0.3.1+cu124" -f https://huggingface.co/datasets/andito/qwentts-cpp-python-wheels/tree/main/whl/cu124
print("파이프라인 설치 완료")

In [ ]:
# 4. RAG 서버용 가상환경 (파이프라인과 의존성 충돌을 막기 위해 분리)
!pip install -q uv
!uv venv -q /content/ragenv --python 3.11
!uv pip install -q --python /content/ragenv/bin/python -r /content/ajou-vis/server/requirements.txt
print("RAG 환경 설치 완료")

In [ ]:
# 5. API 키 설정 + 벡터 DB 구축 (학칙·장학·외국인학생 규정 → ChromaDB)
from google.colab import userdata
key = userdata.get("ANTHROPIC_API_KEY")
with open("/content/ajou-vis/server/.env", "w") as f:
    f.write(f"ANTHROPIC_API_KEY={key}\nLLM_MODEL={LLM_MODEL}\nLLM_EFFORT={LLM_EFFORT}\nGATE_SCORE_THRESHOLD=0.5\n")
del key

%cd /content/ajou-vis
!/content/ragenv/bin/python -c "import sys; sys.path.insert(0,'corpus/scripts'); import embed_corpus as ec; a=ec.load_articles(); d,m,i=ec.build_chunks(a); ec.embed_and_store(d,m,i)"

In [ ]:
# 6. RAG 서버 실행 (백그라운드, 포트 8100)
import subprocess, time, urllib.request, json

def wait_http(url, timeout):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            return urllib.request.urlopen(url, timeout=5).read().decode()
        except Exception:
            time.sleep(2)
    return None

rag_log = open("/content/rag_server.log", "w")
rag_proc = subprocess.Popen(
    ["/content/ragenv/bin/uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8100"],
    cwd="/content/ajou-vis/server", stdout=rag_log, stderr=subprocess.STDOUT)

print("health:", wait_http("http://127.0.0.1:8100/health", 120))
print("warmup:", wait_http("http://127.0.0.1:8100/warmup", 300))   # 임베딩 모델 미리 로드

In [ ]:
# 7. 음성 파이프라인 실행 (serve 모드, 포트 8765)
if STT_BACKEND == "whisper":
    stt_args = ["--stt", "whisper", "--stt_model_name", WHISPER_MODEL,
                "--stt_device", "cuda", "--stt_torch_dtype", "float16", "--language", "auto"]
else:
    stt_args = ["--stt", "qwen3-asr", "--qwen3_asr_model_name", QWEN3_ASR_MODEL,
                "--qwen3_asr_device", "cuda", "--qwen3_asr_language", "auto"]

cmd = ["speech-to-speech", "serve", *stt_args,
       "--llm_backend", "chat-completions",
       "--responses_api_base_url", "http://127.0.0.1:8100/v1",
       "--responses_api_api_key", "dummy", "--model_name", "ajou-vis",
       "--stream_batch_sentences", "1",
       "--tts", "qwen3", "--qwen3_tts_backend", "ggml",
       "--qwen3_tts_ggml_quantization", TTS_QUANT, "--qwen3_tts_device", "cuda",
       "--qwen3_tts_speaker", TTS_SPEAKER, "--qwen3_tts_coalesce_inputs", "False",
       "--host", "127.0.0.1", "--port", "8765"]

pipe_log_path = "/content/pipeline.log"
pipe_log = open(pipe_log_path, "w")
pipe_proc = subprocess.Popen(cmd, cwd="/content/speech-to-speech", stdout=pipe_log, stderr=subprocess.STDOUT)

t0 = time.time()
while time.time() - t0 < 900:   # 첫 실행은 모델 다운로드 때문에 오래 걸림
    text = open(pipe_log_path, errors="replace").read()
    if "Uvicorn running" in text:
        print(f"파이프라인 준비 완료 ({time.time()-t0:.0f}초) — {STT_BACKEND} / TTS {TTS_QUANT}")
        break
    if pipe_proc.poll() is not None:
        print("파이프라인이 종료됨. 로그 마지막 부분:"); print(text[-3000:]); break
    time.sleep(5)
else:
    print("시간 초과. 로그 마지막 부분:"); print(open(pipe_log_path, errors="replace").read()[-3000:])

In [ ]:
# 8. 외부 접속 터널 열기 (cloudflared, 계정 불필요) → Mac 명령 출력
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared

tunnel_log_path = "/content/tunnel.log"
tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8765"],
    stdout=open(tunnel_log_path, "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(tunnel_log_path).read())
    if m:
        url = m.group(0); break
    time.sleep(1)

if url:
    ws = url.replace("https://", "wss://") + "/v1/realtime"
    print("Mac 터미널에서 실행 (헤드폰 권장):\n")
    print(f"    ~/speech-to-speech/run_talk_colab.sh {ws}\n")
    print("주소는 이 셀을 다시 실행할 때마다 바뀐다.")
else:
    print("터널 주소를 못 찾음:"); print(open(tunnel_log_path).read()[-2000:])

In [ ]:
# 9. 상태 확인 — 대화 후 실행: 최근 대화 로그 + 구간별 지연
import subprocess
print("RAG 서버:", "실행 중" if rag_proc.poll() is None else "종료됨",
      "| 파이프라인:", "실행 중" if pipe_proc.poll() is None else "종료됨",
      "| 터널:", "실행 중" if tunnel_proc.poll() is None else "종료됨")
print("\n── 최근 대화 ──")
for line in open(pipe_log_path, errors="replace").read().replace("\r", "\n").splitlines()[-400:]:
    if any(k in line for k in ("USER:", "ASSISTANT: ", "generated", "Last speech detected")):
        print(line[:200])
print("\n── 구간별 지연 ──")
!python /content/ajou-vis/bench/latency_report.py /content/pipeline.log

In [ ]:
# 10. 종료 — 설정(1번 셀)을 바꿔 다시 실험하려면 이 셀 실행 후 6번부터 다시
for p in ("tunnel_proc", "pipe_proc", "rag_proc"):
    if p in globals() and globals()[p].poll() is None:
        globals()[p].terminate()
print("종료 완료")